# 🌿 Análisis Hidráulico de Canales de Riego con Python
## Módulo: Funciones y Estructuras de Control

**Curso:** Fundamentos de Programación 
**Universidad:** Universidad de Sucre 
**Año:** 2026

**Integrantes:**
- De La Ossa Castilla, José David
- Cárdenas Álvarez, Luis
- Noriega Peralta, Luz Dayana
- Triana Palencia, María
- Armesto Bolaño, Maicol
- Clemente De La Cruz, Said

---

## Marco Teórico

### Ecuación de Manning para canal trapezoidal
$$Q = \frac{1}{n} \cdot A \cdot R^{2/3} \cdot S^{1/2}$$

### Geometría trapezoidal
$$A = (b + z \cdot y) \cdot y \qquad P = b + 2y\sqrt{1+z^2} \qquad T = b + 2zy$$

### Número de Froude
$$Fr = \frac{V}{\sqrt{g \cdot D_h}} \qquad D_h = \frac{A}{T} \qquad V = \frac{Q}{A}$$

| Condición | Régimen |
|---|---|
| Fr < 1 | Subcrítico ✅ recomendado para riego |
| Fr = 1 | Crítico |
| Fr > 1 | Supercrítico ⚠️ riesgo de erosión |

### Demanda hídrica de cultivos
$$Q_{cultivo} = \frac{ETo \cdot Kc \cdot A_{cultivo}}{86400}$$


In [ ]:
# Celda 1 — Importaciones
import math
import matplotlib.pyplot as plt
import seaborn as sns

print("="*55)
print("  ANÁLISIS HIDRÁULICO DE CANAL DE RIEGO")
print("="*55)


In [ ]:
# Celda 2 — Funciones geométricas
def calcular_area(b, z, y):
    """Calcula el área de la sección transversal trapezoidal (m²)."""
    return (b + z * y) * y

def calcular_perimetro(b, z, y):
    """Calcula el perímetro mojado del canal trapezoidal (m)."""
    return b + 2 * y * math.sqrt(1 + z**2)

def calcular_ancho_superficial(b, z, y):
    """Calcula el ancho superficial del canal trapezoidal (m)."""
    return b + 2 * z * y

# Prueba
print("Área (b=0.8, z=1.5, y=0.8):", round(calcular_area(0.8, 1.5, 0.8), 3))
print("Perímetro:", round(calcular_perimetro(0.8, 1.5, 0.8), 3))
print("Ancho superficial:", round(calcular_ancho_superficial(0.8, 1.5, 0.8), 3))


In [ ]:
# Celda 3 — Caudal, velocidad y clasificación del régimen
def calcular_caudal(b, z, y, n, S):
    """
    Calcula el caudal y la velocidad en el canal trapezoidal.

    Retorna:
    --------
    tuple : (Q en m³/s, V en m/s)
    """
    if b <= 0 or z < 0 or y <= 0 or n <= 0 or S <= 0:
        raise ValueError("Todos los parámetros deben ser positivos.")
    A = calcular_area(b, z, y)
    P = calcular_perimetro(b, z, y)
    R = A / P
    Q = (1 / n) * A * (R ** (2/3)) * (S ** 0.5)
    V = Q / A
    return Q, V

def clasificar_regimen(Fr):
    """
    Clasifica el régimen de flujo según el número de Froude.

    Retorna:
    --------
    str : nombre del régimen y advertencia si aplica.
    """
    if Fr < 1:
        return "Subcrítico ✅"
    elif Fr == 1:
        return "Crítico"
    else:
        return "Supercrítico ⚠️ riesgo de erosión"

# Prueba
Q_test, V_test = calcular_caudal(0.8, 1.5, 0.8, 0.014, 0.0008)
print(f"Q = {Q_test:.4f} m³/s | V = {V_test:.4f} m/s")


In [ ]:
# Celda 4 — Análisis por rango de tirantes
def analizar_canal(b, z, n, S, y_min, y_max, paso):
    """
    Analiza el canal para un rango de tirantes.

    Retorna:
    --------
    list : lista de diccionarios con resultados por tirante.
    """
    resultados = []
    y = y_min
    g = 9.81
    while y <= y_max + 1e-9:
        A = calcular_area(b, z, y)
        P = calcular_perimetro(b, z, y)
        T = calcular_ancho_superficial(b, z, y)
        R = A / P
        Q, V = calcular_caudal(b, z, y, n, S)
        Dh = A / T
        Fr = V / math.sqrt(g * Dh)
        regimen = clasificar_regimen(Fr)
        resultados.append({
            'y': round(y, 2),
            'A': round(A, 3),
            'P': round(P, 3),
            'R': round(R, 3),
            'Q': round(Q, 4),
            'V': round(V, 3),
            'Fr': round(Fr, 3),
            'regimen': regimen
        })
        y += paso
    return resultados

# Parámetros del canal
b, z, n, S = 0.80, 1.5, 0.014, 0.0008
resultados = analizar_canal(b, z, n, S, 0.20, 1.20, 0.20)

print(f"{'y(m)':>6} {'A(m²)':>7} {'P(m)':>7} {'R(m)':>7} {'Q(m³/s)':>9} {'V(m/s)':>7} {'Fr':>7} {'Régimen':>25}")
print("-" * 80)
for r in resultados:
    print(f"{r['y']:>6.2f} {r['A']:>7.3f} {r['P']:>7.3f} {r['R']:>7.3f} {r['Q']:>9.4f} {r['V']:>7.3f} {r['Fr']:>7.3f} {r['regimen']:>25}")


In [ ]:
# Celda 5 — Demanda de cultivos y verificación de abastecimiento
def calcular_demanda_cultivo(nombre, ETo, Kc, area_ha):
    """
    Calcula la demanda hídrica de un cultivo en m³/s.

    Parámetros:
    -----------
    nombre  : str   — nombre del cultivo
    ETo     : float — evapotranspiración de referencia (mm/día)
    Kc      : float — coeficiente de cultivo
    area_ha : float — área sembrada (ha)

    Retorna:
    --------
    dict : {nombre, Q_demanda}
    """
    area_m2 = area_ha * 10000
    Q_dem = (ETo / 1000) * Kc * area_m2 / 86400
    return {'nombre': nombre, 'Q_demanda': round(Q_dem, 4)}

def verificar_abastecimiento(Q_canal, cultivos):
    """
    Verifica si el caudal del canal abastece cada cultivo.
    """
    print(f"\nCaudal del canal: {Q_canal:.4f} m³/s")
    print(f"{'Cultivo':>10} {'Q demanda (m³/s)':>18} {'Estado':>12}")
    print("-" * 45)
    for c in cultivos:
        estado = "✅ Abastecido" if Q_canal >= c['Q_demanda'] else "❌ Insuficiente"
        print(f"{c['nombre']:>10} {c['Q_demanda']:>18.4f} {estado:>12}")

# Cultivos
cultivos = [
    calcular_demanda_cultivo('Maíz',  5.2, 1.05, 40),
    calcular_demanda_cultivo('Arroz', 6.0, 1.20, 350),
    calcular_demanda_cultivo('Palma', 5.5, 1.10, 600),
    calcular_demanda_cultivo('Caña',  5.8, 1.25, 800),
]

Q_diseno = next(r['Q'] for r in resultados if r['y'] == 0.80)
verificar_abastecimiento(Q_diseno, cultivos)


In [ ]:
# Celda 6 — Reto: tirante mínimo para abastecer todos los cultivos
Q_total_demanda = sum(c['Q_demanda'] for c in cultivos)
print(f"Demanda total de todos los cultivos: {Q_total_demanda:.4f} m³/s")

y_busqueda = 0.20
while True:
    Q_prueba, _ = calcular_caudal(b, z, y_busqueda, n, S)
    if Q_prueba >= Q_total_demanda:
        break
    y_busqueda += 0.01

print(f"Tirante mínimo para abastecer todos los cultivos: y = {y_busqueda:.2f} m")
print(f"Caudal en ese tirante: Q = {Q_prueba:.4f} m³/s")


In [ ]:
# Celda 7 — Visualización
sns.set_theme(style='whitegrid')

tirantes_graf = [r['y'] for r in resultados]
caudales_graf = [r['Q'] for r in resultados]
froudes_graf  = [r['Fr'] for r in resultados]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Gráfica 1: Tirante vs Caudal
ax1.plot(tirantes_graf, caudales_graf, marker='o', color='steelblue', linewidth=2)
ax1.axvline(x=0.80, color='red', linestyle='--', label='y diseño = 0.80 m')
ax1.set_title('Curva de Descarga — Canal de Riego', fontweight='bold')
ax1.set_xlabel('Tirante y (m)')
ax1.set_ylabel('Caudal Q (m³/s)')
ax1.legend()

# Gráfica 2: Tirante vs Número de Froude
ax2.plot(tirantes_graf, froudes_graf, marker='s', color='darkorange', linewidth=2)
ax2.axhline(y=1, color='red', linestyle='--', label='Fr = 1 (crítico)')
ax2.set_title('Número de Froude por Tirante', fontweight='bold')
ax2.set_xlabel('Tirante y (m)')
ax2.set_ylabel('Número de Froude (Fr)')
ax2.legend()

plt.tight_layout()
plt.show()
print("\nEl flujo es subcrítico en todos los tirantes evaluados — condición ideal para riego.")


## 🧠 Reflexión Final

1. **¿Por qué el canal con y = 0.80 m no abastece la caña de azúcar?**  
   Porque la demanda de la caña (800 ha) supera el caudal disponible en ese tirante. Se requiere aumentar el tirante hasta ≈ 0.81 m.

2. **¿Por qué el régimen es subcrítico en todos los casos?**  
   Porque la pendiente S = 0.0008 es muy suave, lo que genera flujos lentos con Fr < 1, ideal para evitar erosión en canales de riego.

3. **¿Qué ventaja tiene usar `while` en el reto del tirante mínimo?**  
   Porque no sabemos de antemano cuántas iteraciones se necesitan — el `while` permite continuar hasta cumplir la condición sin limitar el número de pasos.
